# Regression examples

In [1]:
from sklearn.datasets import load_diabetes
import sys

sys.path.append("..")

from ai_toolkit import (
    BaseDataset, 
    MlTrainerConfig,
    RidgeRegressionModel, 
    XGBoostRegressorModel, 
    get_all_regression_models, 
    RegressionModelTrainer, 
    lazypredict_regression,
    EnsembleVotingRegressorModel,
    EnsembleStackingRegressorModel,
)

## Example data

In [2]:
class RegDataset(BaseDataset):
    """Dataset for diabetes regression task.
    https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html
    """
    def __init__(self):
        """Initialize the clf dataset."""

        super().__init__()

    def load_data(self):
        """Load the diabetes dataset for regression task."""
        
        self.X, self.y = load_diabetes(return_X_y=True, as_frame=True)
        self.X_test = self.X.head()

In [3]:
RDataset = RegDataset()
RDataset.load_data()
RDataset.preprocess()
X_reg, y_reg, X_test_reg = RDataset.get_data()

## Configuration

In [4]:
TrainerConfig = MlTrainerConfig()
TrainerConfig.N_TRAILS = 2
TrainerConfig.EXPERIMENT_NAME = "ml_regression"
TrainerConfig.OPTIMIZE_METRIC = "root_mean_squared_error"

## Training and evaluation

### Train one example model

In [5]:
base_model = RidgeRegressionModel()

# Create a regression model trainer
trainer = RegressionModelTrainer(
    base_model=base_model,
    config=TrainerConfig,
)

In [6]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_reg, 
    y=y_reg, 
    n_trials=TrainerConfig.N_TRAILS, 
)

# y_pred = trainer.predict(X_test)

[I 2025-03-03 20:34:42,558] A new study created in memory with name: Ridge Regressor optimization
[I 2025-03-03 20:34:43,249] Trial 0 finished with value: 165.8331098151396 and parameters: {'alpha': 0.0007481703541112229, 'fit_intercept': False, 'solver': 'saga', 'tol': 1.0287277600293616e-05, 'random_state': 28}. Best is trial 0 with value: 165.8331098151396.
[I 2025-03-03 20:34:43,464] Trial 1 finished with value: 165.83040894830296 and parameters: {'alpha': 0.018956103309033433, 'fit_intercept': False, 'solver': 'svd', 'tol': 0.00027312194549466886, 'random_state': 28}. Best is trial 1 with value: 165.83040894830296.


Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/89 [00:00<?, ?it/s]

  0%|          | 0/89 [00:00<?, ?it/s]

  0%|          | 0/88 [00:00<?, ?it/s]

  0%|          | 0/88 [00:00<?, ?it/s]


# Model: Ridge Regressor

## Best Hyperparameters: {'alpha': 0.018956103309033433, 'fit_intercept': False, 'solver': 'svd', 'tol': 0.00027312194549466886, 'random_state': 28}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 170.1161
Fold 2 Score: 165.4746
Fold 3 Score: 163.0732
Fold 4 Score: 158.9835
Fold 5 Score: 171.5047

## Mean Metrics across all folds:
r2: -3.8281
explained_variance: 0.4426
d2_absoulte_error: -1.4610
d2_tweedie: -3.8281
root_mean_squared_error: 165.8304
median_absolute_error: 151.6746
mean_absolute_percentage_error: 1.2106
mean_absolute_error: 156.0634
mean_squared_error: 27520.7594
max_error: 283.3413
d2_pinball: -1.4610


### Train all regression models

In [7]:
base_models = get_all_regression_models()

for base_model in base_models.values():

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
        n_trials=TrainerConfig.N_TRAILS,
    )

[I 2025-02-27 15:55:29,692] A new study created in memory with name: Ridge Regressor optimization
[I 2025-02-27 15:55:29,743] Trial 0 finished with value: 164.93212911426374 and parameters: {'alpha': 56.89412462271908, 'fit_intercept': False, 'solver': 'cholesky', 'tol': 2.3657744272844202e-05, 'random_state': 28}. Best is trial 0 with value: 164.93212911426374.
[I 2025-02-27 15:55:29,793] Trial 1 finished with value: 55.02297777715465 and parameters: {'alpha': 19.593923527996907, 'fit_intercept': True, 'solver': 'auto', 'tol': 0.00032414972230365187, 'random_state': 28}. Best is trial 1 with value: 55.02297777715465.


Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]

[I 2025-02-27 15:55:49,828] A new study created in memory with name: Bayesian Ridge Regressor optimization



# Model: Ridge Regressor

## Best Hyperparameters: {'alpha': 19.593923527996907, 'fit_intercept': True, 'solver': 'auto', 'tol': 0.00032414972230365187, 'random_state': 28}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 58.0733
Fold 2 Score: 49.3634
Fold 3 Score: 52.8629
Fold 4 Score: 59.3702
Fold 5 Score: 55.4451

## Mean Metrics across all folds:
mean_squared_error: 3040.5436
explained_variance: 0.4705
max_error: 142.8214
root_mean_squared_error: 55.0230
median_absolute_error: 39.4092
r2: 0.4629
d2_pinball: 0.2952
mean_absolute_percentage_error: 0.4006
d2_tweedie: 0.4629
mean_absolute_error: 44.6621
d2_absoulte_error: 0.2952


[I 2025-02-27 15:55:49,879] Trial 0 finished with value: 55.04338605614627 and parameters: {'max_iter': 117, 'tol': 1.2262838086015583e-06, 'alpha_1': 1.228625360042071e-05, 'alpha_2': 3.5074652815200397e-07, 'lambda_1': 2.6753079112109307e-07, 'lambda_2': 2.7781408608618142e-06, 'compute_score': True, 'fit_intercept': True}. Best is trial 0 with value: 55.04338605614627.
[I 2025-02-27 15:55:49,939] Trial 1 finished with value: 55.04338605358529 and parameters: {'max_iter': 398, 'tol': 4.394147449125197e-05, 'alpha_1': 2.1320194471026409e-07, 'alpha_2': 1.2883859417377518e-07, 'lambda_1': 2.7134199094377476e-07, 'lambda_2': 6.405306927355383e-06, 'compute_score': True, 'fit_intercept': True}. Best is trial 1 with value: 55.04338605358529.


...

[LightGBM] [Warning] feature_fraction is set=0.9005412922695842, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9005412922695842
[LightGBM] [Warning] lambda_l1 is set=2.7793627306557007e-08, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.7793627306557007e-08
[LightGBM] [Warning] bagging_fraction is set=0.9943378088269231, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9943378088269231
[LightGBM] [Warning] lambda_l2 is set=0.0265080530542208, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0265080530542208
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.9005412922695842, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9005412922695842
[LightGBM] [Warning] lambda_l1 is set=2.7793627306557007e-08, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.7793627306557007e-08
[LightGBM] [Warning] 


# Model: LightGBM Regressor

## Best Hyperparameters: {'boosting_type': 'dart', 'num_leaves': 48, 'learning_rate': 0.15733838583922927, 'feature_fraction': 0.9005412922695842, 'bagging_fraction': 0.9943378088269231, 'bagging_freq': 3, 'min_child_samples': 77, 'lambda_l1': 2.7793627306557007e-08, 'lambda_l2': 0.0265080530542208, 'device_type': 'cpu', 'verbose': -1, 'seed': 28}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 59.1316
Fold 2 Score: 52.9848
Fold 3 Score: 53.2407
Fold 4 Score: 62.1007
Fold 5 Score: 60.8181

## Mean Metrics across all folds:
mean_squared_error: 3338.7704
explained_variance: 0.4255
max_error: 158.6350
root_mean_squared_error: 57.6552
median_absolute_error: 41.3762
r2: 0.4137
d2_pinball: 0.2669
mean_absolute_percentage_error: 0.3938
d2_tweedie: 0.4137
mean_absolute_error: 46.4827
d2_absoulte_error: 0.2669


[I 2025-02-27 15:57:58,636] A new study created in memory with name: CatBoost Regressor optimization
[I 2025-02-27 15:58:04,862] Trial 0 finished with value: 58.91956402651041 and parameters: {'iterations': 307, 'learning_rate': 0.19943185096877278, 'depth': 4, 'l2_leaf_reg': 0.0001376297282910645, 'bootstrap_type': 'Bayesian', 'random_strength': 6.605502563005976, 'bagging_temperature': 1.4218385163075895, 'od_type': 'Iter', 'od_wait': 28, 'verbose': False, 'random_seed': 28, 'task_type': 'CPU'}. Best is trial 0 with value: 58.91956402651041.
[I 2025-02-27 15:58:17,501] Trial 1 finished with value: 58.101266947531485 and parameters: {'iterations': 829, 'learning_rate': 0.030818951658743578, 'depth': 6, 'l2_leaf_reg': 1.9220872869906462e-07, 'bootstrap_type': 'Bayesian', 'random_strength': 0.1139846995466259, 'bagging_temperature': 9.06463316874607, 'od_type': 'Iter', 'od_wait': 33, 'verbose': False, 'random_seed': 28, 'task_type': 'CPU'}. Best is trial 1 with value: 58.101266947531485

Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]


# Model: CatBoost Regressor

## Best Hyperparameters: {'iterations': 829, 'learning_rate': 0.030818951658743578, 'depth': 6, 'l2_leaf_reg': 1.9220872869906462e-07, 'bootstrap_type': 'Bayesian', 'random_strength': 0.1139846995466259, 'bagging_temperature': 9.06463316874607, 'od_type': 'Iter', 'od_wait': 33, 'verbose': False, 'random_seed': 28, 'task_type': 'CPU'}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 59.0442
Fold 2 Score: 56.0719
Fold 3 Score: 54.5014
Fold 4 Score: 64.9027
Fold 5 Score: 55.9862

## Mean Metrics across all folds:
mean_squared_error: 3389.4971
explained_variance: 0.4032
max_error: 173.5286
root_mean_squared_error: 58.1013
median_absolute_error: 40.2267
r2: 0.3976
d2_pinball: 0.2692
mean_absolute_percentage_error: 0.4007
d2_tweedie: 0.3976
mean_absolute_error: 46.2405
d2_absoulte_error: 0.2692


In [8]:
df_mean_results = lazypredict_regression(
    X=X_reg, 
    y=y_reg, 
    n_splits=TrainerConfig.N_SPLITS,
    random_state=TrainerConfig.RANDOM_STATE,
)

df_mean_results

Cross-Validation:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 42/42 [00:03<00:00, 11.16it/s]


,Adjusted R-Squared,R-Squared,RMSE,Time Taken,Adjusted R-Squared Std
Model,,,,,
PoissonRegressor,0.40,0.47,54.79,0.02,0.14
LassoLars,0.39,0.46,54.95,0.02,0.14
Lasso,0.39,0.46,54.95,0.02,0.14
ElasticNetCV,0.39,0.46,55.06,0.13,0.13
BayesianRidge,0.39,0.46,55.04,0.02,0.13
SGDRegressor,0.39,0.46,55.06,0.02,0.14
LassoLarsIC,0.39,0.46,55.07,0.02,0.14
Ridge,0.39,0.46,55.06,0.02,0.14
LinearRegression,0.39,0.46,55.11,0.01,0.15


### Ensemble

In [9]:
# (Model, mlflow run_id) pairs
models = [
    (RidgeRegressionModel(), "6a35b548ae5048b2adcfccdd4dd87165"),
    (XGBoostRegressorModel(), "84ddfd37c1044ea59ea7c736c80a61e6"),
]

meta_model = RidgeRegressionModel()

#### Voting

In [10]:
base_model = EnsembleVotingRegressorModel(
    models=models,
)

# Create a regression model trainer
trainer = RegressionModelTrainer(
    base_model=base_model,
    config=TrainerConfig,
)

In [11]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_reg, 
    y=y_reg, 
    n_trials=TrainerConfig.N_TRAILS, 
)

# y_pred = trainer.predict(X_test)

[I 2025-02-27 15:59:49,619] A new study created in memory with name: Ensemble Voting Regressor optimization


[15:59:49] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:50] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:51] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:52] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:53] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.



[I 2025-02-27 15:59:53,620] Trial 0 finished with value: 95.2979997055374 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=19.593923527996907, random_state=28, tol=0.00032414972230365187)), ('XGBoost Regressor', XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9605608847616904, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.004041224828534168,
             gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.0071810902163983985,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             n_estimators=588, n_jobs=None, num_parallel_tree=None,
             obje

[15:59:53] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:54] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:54] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:55] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[15:59:56] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.



[I 2025-02-27 15:59:57,231] Trial 1 finished with value: 144.4786430022477 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=19.593923527996907, random_state=28, tol=0.00032414972230365187)), ('XGBoost Regressor', XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9605608847616904, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.004041224828534168,
             gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.0071810902163983985,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             n_estimators=588, n_jobs=None, num_parallel_tree=None,
             obj

Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]

[15:59:57] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:00] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:03] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:05] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:08] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.




# Model: Ensemble Voting Regressor

## Best Hyperparameters: {'estimators': [('Ridge Regressor', Ridge(alpha=19.593923527996907, random_state=28, tol=0.00032414972230365187)), ('XGBoost Regressor', XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9605608847616904, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.004041224828534168,
             gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.0071810902163983985,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             n_estimators=588, n_jobs=None, num_parallel_tree=None,
             objective='reg:squaredlogerror',

#### Stacking

In [12]:
base_model = EnsembleStackingRegressorModel(
    models=models,
    meta_model=meta_model,
)

# Create a regression model trainer
trainer = RegressionModelTrainer(
    base_model=base_model,
    config=TrainerConfig,
)

In [13]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_reg, 
    y=y_reg, 
    n_trials=TrainerConfig.N_TRAILS, 
)

# y_pred = trainer.predict(X_test)

[I 2025-02-27 16:00:20,579] A new study created in memory with name: Ensemble Stacking Regressor optimization


[16:00:20] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:21] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:21] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:22] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:22] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:22] WARNING: 

[I 2025-02-27 16:00:33,940] Trial 0 finished with value: 59.65989919610875 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=19.593923527996907, random_state=28, tol=0.00032414972230365187)), ('XGBoost Regressor', XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9605608847616904, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.004041224828534168,
             gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.0071810902163983985,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             n_estimators=588, n_jobs=None, num_parallel_tree=None,
             obj

[16:00:33] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:34] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:34] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:35] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:35] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:36] WARNING: 

[I 2025-02-27 16:00:47,010] Trial 1 finished with value: 59.65989919610875 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=19.593923527996907, random_state=28, tol=0.00032414972230365187)), ('XGBoost Regressor', XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9605608847616904, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.004041224828534168,
             gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.0071810902163983985,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             n_estimators=588, n_jobs=None, num_parallel_tree=None,
             obj

Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]

[16:00:47] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:47] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:48] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:48] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:49] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-0fc7796c793e6356f-1/xgboost/xgboost-ci-windows/src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:00:49] WARNING: 


# Model: Ensemble Stacking Regressor

## Best Hyperparameters: {'estimators': [('Ridge Regressor', Ridge(alpha=19.593923527996907, random_state=28, tol=0.00032414972230365187)), ('XGBoost Regressor', XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9605608847616904, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.004041224828534168,
             gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.0071810902163983985,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             n_estimators=588, n_jobs=None, num_parallel_tree=None,
             objective='reg:squaredlogerror